# 07 - Embedding dan Faiss

In [1]:
!pip install -q faiss-cpu sentence-transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.0 MB/s eta 0:00:00


In [2]:
import json
import glob
import os
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi
import hashlib

## 1. Konfigurasi

In [3]:
# CONFIG 
CHUNKS_DIR       = "/kaggle/input/notebooks/muhammadnabilul/04-chunking-and-splitting/chunks"
OUTPUT_DIR       = "/kaggle/working/rag_knowledge_base"
HF_REPO_ID       = "Makaareeem/publikasi-rag-knowledge-base"
 
MODEL_ID         = "nomic-ai/nomic-embed-text-v1.5"
TASK_PREFIX      = "search_document: "
BATCH_SIZE       = 64
EMBED_DIM        = 768
 
CHUNK_META_PATH  = f"{OUTPUT_DIR}/chunk_metadata.jsonl"
EMBEDDINGS_PATH  = f"{OUTPUT_DIR}/embeddings.npy"
FAISS_INDEX_PATH = f"{OUTPUT_DIR}/faiss_index.bin"
MANIFEST_PATH    = f"{OUTPUT_DIR}/index_manifest.json"
 
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# I/O
def load_all_chunks(chunks_dir):
    chunks = []
    for fpath in sorted(glob.glob(f"{chunks_dir}/*.json")):
        with open(fpath, "r", encoding="utf-8") as f:
            doc_chunks = json.load(f)
        chunks.extend(doc_chunks)
    return chunks
 
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

## 2. Fungsi

### a. metadata

In [5]:
def build_metadata_store(chunks):
    metadata = []
    for faiss_id, chunk in enumerate(chunks):
        record = {
            "faiss_id":         faiss_id,
            "chunk_id":         chunk["chunk_id"],
            "text":             chunk["text"],
            "document_title":   chunk.get("document_title", ""),
            "document_year":    chunk.get("document_year", ""),
            "source_file":      chunk.get("source_file", ""),
            "bps_url":          chunk.get("bps_url", ""),
            "section_id":       chunk.get("section_id", ""),
            "section_title":    chunk.get("section_title", ""),
            "section_level":    chunk.get("section_level", None),
            "page_start":       chunk.get("page_start", None),
            "page_end":         chunk.get("page_end", None),
            "source_pages":     chunk.get("source_pages", []),
            "chunk_type":       chunk.get("chunk_type", ""),
            "token_estimate":   chunk.get("token_estimate", None),
        }
        metadata.append(record)
    return metadata

### b. embed

In [6]:
def embed_chunks(chunks, model, batch_size=64):
    texts = [TASK_PREFIX + chunk["text"] for chunk in chunks]
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return embeddings.astype(np.float32)

### c. faiss

In [7]:
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

In [8]:
def compute_manifest(chunks, embeddings):
    return {
        "total_chunks":    len(chunks),
        "embedding_dim":   int(embeddings.shape[1]),
        "index_type":      "IndexFlatIP",
        "model_id":        MODEL_ID,
        "task_prefix":     TASK_PREFIX,
        "normalized":      True,
        "pipeline_version": "embedding_v1",
        "embeddings_sha256": hashlib.sha256(embeddings.tobytes()).hexdigest(),
    }

## 3. Pipeline

In [9]:
print("Loading chunks...")
chunks = load_all_chunks(CHUNKS_DIR)
print(f"  Total chunks loaded: {len(chunks)}")
 
print("Building metadata store...")
metadata = build_metadata_store(chunks)
save_jsonl(metadata, CHUNK_META_PATH)
print(f"  Saved: {CHUNK_META_PATH}")
 
print("Loading embedding model...")
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_Faiss")
login(token=hf_token)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

model = SentenceTransformer(
    MODEL_ID,
    trust_remote_code=True,
    device=DEVICE,
)
 
print("Embedding chunks...")
embeddings = embed_chunks(chunks, model, batch_size=BATCH_SIZE)
np.save(EMBEDDINGS_PATH, embeddings)
print(f"  Saved: {EMBEDDINGS_PATH}  shape={embeddings.shape}")

print("Validating embeddings...")
norms = np.linalg.norm(embeddings, axis=1)
assert embeddings.ndim == 2,             f"Expected 2D, got {embeddings.ndim}D"
assert embeddings.shape[1] == EMBED_DIM, f"Dim mismatch: {embeddings.shape[1]}"
assert not np.isnan(embeddings).any(),   "NaN detected"
assert not np.isinf(embeddings).any(),   "Inf detected"
assert np.allclose(norms, 1.0, atol=1e-5), f"Not normalized — norm range: [{norms.min():.6f}, {norms.max():.6f}]"
print(f"  Validation passed — shape={embeddings.shape}, norm_mean={norms.mean():.6f}")

print("Building FAISS index...")
index = build_faiss_index(embeddings)
faiss.write_index(index, FAISS_INDEX_PATH)
print(f"  Saved: {FAISS_INDEX_PATH}  total_vectors={index.ntotal}")

Loading chunks...
  Total chunks loaded: 23336
Building metadata store...
  Saved: /kaggle/working/rag_knowledge_base/chunk_metadata.jsonl
Loading embedding model...
Device: cuda


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Embedding chunks...


Batches:   0%|          | 0/365 [00:00<?, ?it/s]

  Saved: /kaggle/working/rag_knowledge_base/embeddings.npy  shape=(23336, 768)
Validating embeddings...
  Validation passed — shape=(23336, 768), norm_mean=1.000000
Building FAISS index...
  Saved: /kaggle/working/rag_knowledge_base/faiss_index.bin  total_vectors=23336


## 4. Summary

In [10]:
manifest = compute_manifest(chunks, embeddings)
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)
 
print(" Index Manifest ")
for k, v in manifest.items():
    print(f"  {k}: {v}")

 Index Manifest 
  total_chunks: 23336
  embedding_dim: 768
  index_type: IndexFlatIP
  model_id: nomic-ai/nomic-embed-text-v1.5
  task_prefix: search_document: 
  normalized: True
  pipeline_version: embedding_v1
  embeddings_sha256: 2fe526202966c88bf29e61a8560926ac30868d28325e29f26efd6cfccc38da7d


In [11]:
index_loaded  = faiss.read_index(FAISS_INDEX_PATH)
emb_loaded    = np.load(EMBEDDINGS_PATH)
with open(CHUNK_META_PATH) as f:
    meta_lookup = [json.loads(line) for line in f]

print(f"index.ntotal  : {index_loaded.ntotal}")
print(f"embeddings    : {emb_loaded.shape}")
print(f"metadata rows : {len(meta_lookup)}")

TEST_QUERY = "search_query: Berapa tingkat pengangguran terbuka Indonesia?"
q_vec = model.encode([TEST_QUERY], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
scores, ids = index_loaded.search(q_vec, 5)

print(f"\nTop-5 hasil — '{TEST_QUERY}'")
for score, fid in zip(scores[0], ids[0]):
    c = meta_lookup[fid]
    print(f"  [{score:.4f}] {c['chunk_id']}")
    print(f"           {c['document_title']} ({c['document_year']}) | hal {c['page_start']}")
    print(f"           {c['text'][:100]}...")

index.ntotal  : 23336
embeddings    : (23336, 768)
metadata rows : 23336

Top-5 hasil — 'search_query: Berapa tingkat pengangguran terbuka Indonesia?'
  [0.7830] statistik-pemuda-indonesia-2025__sec_066__c001
           Statistik Pemuda Indonesia 2025 (2025) | hal 47
           Tingkat Pengangguran Terbuka (TPT)
adalah persentase angkatan kerja yang tidak mempunyai pekerjaan, ...
  [0.7824] statistik-pemuda-indonesia-2024__sec_064__c001
           Statistik Pemuda Indonesia 2024 (2024) | hal 55
           Tingkat Pengangguran Terbuka (TPT)
adalah persentase angkatan kerja yang tidak mempunyai pekerjaan, ...
  [0.7765] statistik-pemuda-indonesia-2023__5__c007
           Statistik Pemuda Indonesia 2023 (2023) | hal 263
           pengangguran terdidik. Tingginya tingkat pengangguran pada pemuda terdidik tersebut menunjukkan adan...
  [0.7592] statistik-pendidikan-2023__sec_041__c001
           Statistik Pendidikan 2023 (2023) | hal 196
           B. Pendidikan Tertinggi yang Ditamatkan
T

## 5. Push to HF

In [12]:
print("\nPushing artefacts to HuggingFace Hub...")
api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
 
for fpath in [CHUNK_META_PATH, EMBEDDINGS_PATH, FAISS_INDEX_PATH, MANIFEST_PATH]:
    api.upload_file(
        path_or_fileobj=fpath,
        path_in_repo=Path(fpath).name,
        repo_id=HF_REPO_ID,
        repo_type="dataset",
    )
    print(f"  Uploaded: {Path(fpath).name}")
 
print("\nNB07 complete.")


Pushing artefacts to HuggingFace Hub...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


  Uploaded: chunk_metadata.jsonl


...

  Uploaded: embeddings.npy


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  Uploaded: faiss_index.bin
  Uploaded: index_manifest.json

NB07 complete.
